# Create bar charts comparing Bragg diffraction properties for ALS crystals

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

df = pd.read_csv('source_data/dials_stats_humidity.csv').set_index(['date','crystal'])
df

In [ ]:
# assign names to each row

xtal_mapping = {
    'xtal3':'no sleeve + paratone',
    'xtal7':'sleeve + paratone',
    'xtal9':'sleeve + no oil',
    'xtal10':'sleeve + santovac',
}

def dataset_name(row):
    date, crystal = row.name
    if date != '2025-07-01':
        return date
    else:
        return xtal_mapping[crystal]

df['name'] = df.apply(dataset_name,axis=1)
df

In [ ]:
df_tmp = df.reset_index(drop=True).set_index('name')
for c in ['Mosaicity','a','c']:
    df_tmp[c] = df_tmp[c].fillna(df_tmp.groupby(df_tmp.index)[c].transform('mean'))
df_tmp = df_tmp.dropna(subset=['Wilson B'])
df_tmp

In [ ]:
col_order = ['Wilson B','Mosaicity','a','c']
row_order = ['2025-10-15','2025-11-25','sleeve + no oil','sleeve + santovac','sleeve + paratone','no sleeve + paratone']
bar_colors = ['gray','gray','C0','C2','C1','C3']

fig = plt.figure(figsize=(7, 3))
gs = gridspec.GridSpec(2, 4, height_ratios=[0.1, 1], hspace=0.02, wspace=0.2)
axes = []
for j in range(4):
    if j == 0:
        axes.append(fig.add_subplot(gs[1,0]))
    else:
        axes.append(fig.add_subplot(gs[1,j], sharey=axes[0]))

df_tmp.reindex(row_order)[col_order].plot.barh(
    subplots=True,
    ax=axes,
    width=0.8,
    color='blue', 
    legend=False, 
    ylabel='')

df_ref = df_tmp.loc[['2025-10-15','2025-11-25']]

for name, ax in zip(col_order,axes):
    if name=='Wilson B':
        ax.set_xlim(10,22)
        ax.set_xlabel('Wilson B (Å$^2$)')
    elif name=='Mosaicity':
        ax.set_xlim(0,0.2)
        ax.set_xlabel('Mosaicity (deg.)')
    elif name=='a':
        ax.set_xlabel('Unit cell a=b (Å)')
        ax.set_xlim(88.9,89.3)
    elif name=='c':
        ax.set_xlim(39.4,40.4)
        ax.set_xlabel('Unit cell c (Å)')
    ax.set_title(None)
    ref_val = df_ref[name].mean()
    ax.axvline(ref_val, linestyle='--', color='k')
    for j, color in enumerate(bar_colors):
        ax.get_children()[j].set_color(color)

ax1 = fig.add_subplot(gs[0,0:2])
ax1.set_xlim(-0.25,1.25)
ax1.set_title('Crystal disorder')

ax2 = fig.add_subplot(gs[0,2:4])
ax2.set_xlim(1.25,-0.25)
ax2.set_title('Unit cell contraction')

for ax in [ax1,ax2]:
    ax.fill([0,1,1],[0.5,0.2,0.8],color='k',zorder=3,clip_on=False)
    ax.text(1.125, 0.5, 'more', ha='center', va='center')
    ax.text(-.125, 0.5, 'less', ha='center', va='center')
    ax.set_ylim(0,1)
    ax.axis('off')

plt.savefig("fig4_charts.svg", transparent=True, bbox_inches="tight")
plt.savefig("fig4_charts.pdf", transparent=True, bbox_inches="tight")
plt.savefig("fig4_charts.png", transparent=True, dpi=300, bbox_inches="tight")